Code to format measured data obtained from pictures of curves in connnection sheets.

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- INPUT FILES ---
folder = "/home/philinux/model_validation_remy/dynamic-model-validation-open-engine/test_cases/test_case_cernay/Test_case_Cernay/I5/Measured_data"
file_u = folder + "/P curve.csv"
file_q = folder + "/Q curve.csv"

# --- OUTPUT FILE ---
output_file = folder + "/measured_data.csv"

Snom = 1/0.943
Unom = 0.766/0.943
DeltaT = 19

# Load CSVs (automatically detect separators)
df_u = pd.read_csv(file_u, sep=";", decimal=",", engine="python")
df_q = pd.read_csv(file_q, sep=";", decimal=",", engine="python")

# Assume first column is time
time_u = df_u.iloc[:, 0].values
U_values = df_u.iloc[:, 1].values/Unom

time_q = df_q.iloc[:, 0].values
Q_values = df_q.iloc[:, 1].values/Snom

# Shift both times by DeltaT seconds
time_u = time_u + DeltaT
time_q = time_q + DeltaT

# Shift ordinates
# Q_values = Q_values - 0.0899

# Create a common time axis between min and max of both
t_min = min(time_u.min(), time_q.min())
t_max = max(time_u.max(), time_q.max())

# But user wants exactly from 0 to 100 s
common_time = np.linspace(0, 30, 5000)  # very smooth interpolation grid

# Interpolate U and Q
U_interp = np.interp(common_time, time_u, U_values, left=U_values[0], right=U_values[-1])
Q_interp = np.interp(common_time, time_q, Q_values, left=Q_values[0], right=Q_values[-1])

# Build output DataFrame
df_out = pd.DataFrame({
    "time": common_time,
    "Bess_BESS_WTGTerminalMeasurements_PPu": U_interp,
    "Bess_BESS_WTGTerminalMeasurements_QPu": Q_interp
})

# Save
df_out.to_csv(output_file, index=False)

print(f"Fichier généré : {output_file}")


Fichier généré : /home/philinux/model_validation_remy/dynamic-model-validation-open-engine/test_cases/test_case_cernay/Test_case_Cernay/I5/Measured_data/measured_data.csv


In [50]:
import pandas as pd
import plotly.graph_objects as go

# Charger les données
df = pd.read_csv(folder + "/measured_data.csv")

time = df["time"]
U = df["Bess_BESS_WTGTerminalMeasurements_PPu"]
Q = df["Bess_BESS_WTGTerminalMeasurements_QPu"]

#Bess_BESS_WTGTerminalMeasurements_PPu,Bess_BESS_WTGTerminalMeasurements_QPu
#Wind_Turbine_WPP_wPPControl_measurements_UPu,Wind_Turbine_WPP_wPPControl_measurements_QPu

# --- Courbe interactive U ---
fig_U = go.Figure()
fig_U.add_trace(go.Scatter(x=time, y=U, mode='lines', name='U'))
fig_U.update_layout(
    title="U Curve (measured_data)",
    xaxis_title="Time (s)",
    yaxis_title="U Pu"
)
fig_U.show()

# --- Courbe interactive Q ---
fig_Q = go.Figure()
fig_Q.add_trace(go.Scatter(x=time, y=Q, mode='lines', name='Q'))
fig_Q.update_layout(
    title="Q Curve (measured_data)",
    xaxis_title="Time (s)",
    yaxis_title="Q Pu"
)
fig_Q.show()
